In [1]:
from pathlib import Path
import os
import multiprocessing
import re
import numpy as np
import pandas as pd
import xarray as xr
from joblib import Parallel, delayed, parallel_backend

COORDINATE_NAMES = {
    'lat': ('lat', 'latitude', 'Latitude', 'LATITUDE', 'lat_FULL', 'Y', 'y'),
    'lon': ('lon', 'longitude', 'Longitude', 'LONGITUDE', 'lon_FULL', 'X', 'x'),
    'time': ('time', 'time_counter', 'Time', 'TIME'),
}


def normalize_coordinates(ds):
    """Standardize a rectilinear degree grid; keep data attached while sorting."""
    rename = {}
    for target, candidates in COORDINATE_NAMES.items():
        source = next((name for name in candidates if name in ds.variables), None)
        if source is None:
            raise ValueError(f'Missing {target} coordinate')
        if source != target:
            rename[source] = target
    ds = ds.rename(rename)
    for axis in ('lat', 'lon'):
        coord = ds[axis]
        if coord.ndim != 1:
            raise ValueError(f'{axis}: only 1-D rectilinear grids are supported')
        units = str(coord.attrs.get('units', '')).lower()
        if 'degree' not in units:
            raise ValueError(f'{axis}: expected degree units, found {units!r}')
        if coord.dims != (axis,):
            ds = ds.swap_dims({coord.dims[0]: axis})
        values = ds[axis].values
        if values.size < 2 or not np.isfinite(values).all():
            raise ValueError(f'{axis}: coordinates must contain finite grid centers')
    if np.any(np.abs(ds.lat.values) > 90):
        raise ValueError('Latitude outside [-90, 90]; do not wrap latitude')
    lon = ds.lon.values
    if not (np.all((lon >= -180) & (lon <= 180)) or
            np.all((lon >= 0) & (lon <= 360))):
        raise ValueError('Unrecognized longitude range; inspect this grid')
    if np.any(lon >= 180):
        attrs = dict(ds.lon.attrs)
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180)
        ds.lon.attrs = attrs
    ds = ds.sortby(['lat', 'lon'])
    for axis in ('lat', 'lon'):
        if np.any(np.diff(ds[axis].values) <= 0):
            raise ValueError(f'{axis}: duplicate coordinates after normalization')
    return ds


def year_labels(time):
    """Extract years without pretending that all monthly files use 360-day years."""
    units = str(time.attrs.get('units', '')).strip()
    calendar = str(time.attrs.get('calendar', 'standard')).lower()
    values = np.asarray(time.values)
    match = re.fullmatch(r'months since\s+(\d+)-\d+-\d+(?:[ T].*)?', units)
    if match:
        if not np.isfinite(values).all() or not np.allclose(values, np.rint(values)):
            raise ValueError('Fractional months require an explicit calendar interpretation')
        origin_month = int(units.split('since', 1)[1].strip().split('-')[1])
        return int(match[1]) + (origin_month - 1 + np.rint(values).astype(int)) // 12
    if calendar in ('noleap', '365_day') and units.startswith('years since'):
        units = units.replace('years since', 'common_years since', 1)
    from netCDF4 import num2date
    return np.array([d.year for d in num2date(values, units=units, calendar=calendar)])


def cell_areas(ds):
    """Spherical areas for this notebook's global, regularly spaced longitude grids."""
    lat = ds.lat.values.astype(float)
    lon = ds.lon.values.astype(float)
    spacing = np.diff(lon)
    dlon = float(np.median(spacing))
    if not np.allclose(spacing, dlon, rtol=1e-4, atol=1e-5):
        raise ValueError('Irregular longitude spacing needs explicit cell bounds')
    if not np.isclose(dlon * len(lon), 360, atol=1e-3):
        raise ValueError('Expected a global longitude grid')
    bounds_name = ds.lat.attrs.get('bounds')
    valid_bounds = False
    if bounds_name and bounds_name in ds:
        bounds = ds[bounds_name].transpose('lat', ...).values
        if bounds.shape == (len(lat), 2) and np.isfinite(bounds).all():
            lower, upper = np.min(bounds, axis=1), np.max(bounds, axis=1)
            valid_bounds = bool(
                np.all(upper > lower)
                and np.all(lower >= -90) and np.all(upper <= 90)
                and np.all(lower <= lat) and np.all(lat <= upper))
        if not valid_bounds:
            if not np.allclose(np.diff(lat), np.median(np.diff(lat)),
                               rtol=1e-4, atol=1e-5):
                raise ValueError('Invalid latitude bounds on an irregular grid; '
                                 'inspect the source before estimating areas')
            import warnings
            warnings.warn(
                f'{bounds_name}: invalid latitude bounds; deriving bounds '
                'from the regular latitude centers.', RuntimeWarning)
    if not valid_bounds:
        edges = np.r_[lat[0] - (lat[1] - lat[0]) / 2,
                      (lat[:-1] + lat[1:]) / 2,
                      lat[-1] + (lat[-1] - lat[-2]) / 2]
        lower, upper = edges[:-1], edges[1:]
    lower, upper = np.clip(lower, -90, 90), np.clip(upper, -90, 90)
    area = 6371000.0**2 * np.deg2rad(dlon) * (
        np.sin(np.deg2rad(upper)) - np.sin(np.deg2rad(lower)))
    if not np.isfinite(area).all() or np.any(area <= 0):
        raise ValueError('Cell areas must be finite and strictly positive')
    return xr.DataArray(area, dims='lat', coords={'lat': ds.lat})


def nearest_cells(ds, sites):
    lat_points = sites['Lat.'].to_numpy(dtype=float)
    lon_points = sites['Lon.'].to_numpy(dtype=float)
    if not np.isfinite(lat_points).all() or not np.isfinite(lon_points).all():
        raise ValueError('Missing or non-finite site coordinates')
    if np.any(np.abs(lat_points) > 90) or np.any(np.abs(lon_points) > 180):
        raise ValueError('Expected site latitude [-90, 90] and longitude [-180, 180]')
    lat = ds.lat.values
    lower = max(-90, lat[0] - (lat[1] - lat[0]) / 2)
    upper = min(90, lat[-1] + (lat[-1] - lat[-2]) / 2)
    if np.any((lat_points < lower) | (lat_points > upper)):
        raise ValueError('A site is outside the model latitude coverage')
    iy = np.abs(lat[:, None] - lat_points).argmin(axis=0)
    angular_distance = np.abs((ds.lon.values[:, None] - lon_points + 180) % 360 - 180)
    ix = angular_distance.argmin(axis=0)
    pairs = np.unique(np.column_stack([iy, ix]), axis=0)
    return {axis: xr.DataArray(pairs[:, j], dims='cell')
            for j, axis in enumerate(('lat', 'lon'))}


def extract_yearly_cells(ds, variable, sites, start_year=None):
    ds = normalize_coordinates(ds)
    if set(ds[variable].dims) != {'time', 'lat', 'lon'}:
        raise ValueError(f'{variable}: expected only time, lat, lon dimensions')
    years = year_labels(ds.time)
    indexers = nearest_cells(ds, sites)
    selected = ds[variable].isel(indexers).assign_coords(year=('time', years))
    if start_year is not None:
        selected = selected.isel(time=np.flatnonzero(years >= start_year))
    if selected.sizes['time'] == 0:
        raise ValueError('No time steps in the requested period')

    selected = selected.load()
    annual = selected.groupby('year').mean('time', skipna=True)
    weighted = annual * cell_areas(ds).isel(lat=indexers['lat'])
    frame = weighted.rename(variable).to_dataframe().reset_index()
    return frame[['year', 'lat', 'lon', variable]]


In [10]:
PROJECT = #Enter the project path with downloaded TRENDYv10 data under DATA_DIR
DATA_DIR = PROJECT / 'data/TRENDYv10/downloads'
SITE_FILE = PROJECT / 'data/cabon/Cabonetal_site_info.csv'
# Overwrite matching results by default; an environment override supports staging.
OUTPUT_DIR = PROJECT '/results/TRENDYv10/31_site_weighted'

MODELS = ["CABLE-POP", "ISBA-CTRIP", "CLM5.0", "LPJ-GUESS", "LPX-Bern", "CLASSIC", "CLASSIC-N", "ORCHIDEE", "ORCHIDEEv3"]
           
SCENARIOS = ['S0', 'S1']
VARIABLES = ['cLeaf', 'cRoot', 'cWood', 'gpp', 'ra']
START_YEAR = None 

# Concurrent files; lower this if memory or disk throughput is limited.
# Set num_cores = 1 for a sequential run.
num_cores = multiprocessing.cpu_count() - 1


## Audit all input coordinate ranges

In [ ]:
# Only coordinate arrays are read here, not the flux fields.
coordinate_rows = []
for model in MODELS:
    for scenario in SCENARIOS:
        for variable in VARIABLES:
            path = DATA_DIR / model / f'{model}_{scenario}_{variable}.nc'
            with xr.open_dataset(path, decode_times=False) as raw:
                row = dict(model=model, scenario=scenario, variable=variable)
                for axis in ('lat', 'lon'):
                    name = next(n for n in COORDINATE_NAMES[axis] if n in raw.variables)
                    values = raw[name].values
                    row[f'{axis}_min'] = float(values.min())
                    row[f'{axis}_max'] = float(values.max())
                    row[f'{axis}_order'] = ('ascending' if np.all(np.diff(values) > 0)
                                           else 'descending' if np.all(np.diff(values) < 0)
                                           else 'nonmonotonic')
                checked = normalize_coordinates(raw)
                row['convert_longitude'] = row['lon_max'] >= 180
                coordinate_rows.append(row)
coordinate_audit = pd.DataFrame(coordinate_rows)
display(coordinate_audit.drop(columns=['scenario', 'variable']).drop_duplicates())


In [ ]:
sites = pd.read_csv(SITE_FILE)
sites = sites.loc[sites['On-site RW'].eq(True)].copy()
if sites.empty:
    raise ValueError('No sites selected by On-site RW')


def process_and_save_variable(model, scenario, variable, data_dir, output_root,
                              sites, start_year):
    """Read, extract, and save one file within an isolated worker process."""
    data_dir = Path(data_dir)
    output_root = Path(output_root)
    output_dir = output_root / model
    output_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / model / f'{model}_{scenario}_{variable}.nc'
    
    import time
    started = time.monotonic()
    print(f'Starting {model} {scenario} {variable}', flush=True)
    try:
        with xr.open_dataset(path, decode_times=False) as ds:
            frame = extract_yearly_cells(ds, variable, sites, start_year)
    except Exception as exc:
        raise RuntimeError(f'Extraction failed for {path}') from exc
    output_file = output_dir / f'{variable}_{scenario}_31_site_weighted_yearly_mean.csv'
    temporary_file = output_file.with_suffix('.csv.tmp')
    frame.to_csv(temporary_file, index=False)
    os.replace(temporary_file, output_file)
    message = (f'Saved {model} {scenario} {variable}: {len(frame):,} rows '
               f'in {time.monotonic() - started:.1f}s')
    print(message, flush=True)
    return message

with parallel_backend('loky', inner_max_num_threads=1):
    saved_messages = Parallel(n_jobs=num_cores, batch_size=1,
                              pre_dispatch=num_cores, verbose=50)(
        delayed(process_and_save_variable)(
            model, scenario, variable, DATA_DIR, OUTPUT_DIR, sites, START_YEAR)
        for model in MODELS
        for scenario in SCENARIOS
        for variable in VARIABLES
    )
for message in saved_messages:
    print(message)

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)